# Asistente Inteligente (RAG)
Este notebook usará los chunks almacenados para brindarle contexto al LLM, el cual evaluará y dará recomendaciones sobre los requerimientos que escribamos.

In [1]:
!source env/bin/activate
%pip install -q langchain langchain-community langchain-openai psycopg2-binary pgvector tiktoken

Note: you may need to restart the kernel to use updated packages.


## Configurar variables de entorno y LLM

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores.pgvector import PGVector

load_dotenv()

db_name = os.getenv("POSTGRES_DB")
db_password = os.getenv("POSTGRES_PASSWORD")
db_user = os.getenv("POSTGRES_USER")
db_host = os.getenv("POSTGRES_HOST", "localhost")
db_port = os.getenv("POSTGRES_PORT", "5432")

CONNECTION_STRING = f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
COLLECTION_NAME = "iso_standards"

# componentes principales
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.1) # Temperatura baja para que sea conciso y certero

# vectorial db conection
vector_store = PGVector(
    connection_string=CONNECTION_STRING,
    embedding_function=embeddings,
    collection_name=COLLECTION_NAME
)


## Definir el Prompt para el Evaluador

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template = """
Eres un experto en Ingeniería de Requisitos en una pasantía de investigación.
Se te proporcionará contexto basado en directrices y estándares para escribir requerimientos.
Tu tarea es recibir un requerimiento de un usuario, evaluarlo, y sugerir cómo mejorarlo basándote en el contexto.

Reglas:
1. Di en qué falla el requerimiento original y sé específico (ej. falta de métricas, ambigüedad).
2. Da recomendaciones para mejorarlo.
3. Presenta una versión reescrita y óptima del requerimiento.

Contexto normativo: {context}

Requerimiento propuesto: {requirement}
"""

prompt = ChatPromptTemplate.from_template(template)

## Cadena RAG y Pruebas

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# configurar la cadena para que recupere los k trozos con mayor similitud a la consulta
k = 3

retriever = vector_store.as_retriever(search_kwargs={"k": k})

chain = (
    {"context": retriever, "requirement": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# hacer la prueba con un requerimiento deficiente e impreciso
requerimiento_usuario = "El sistema de base de datos deberá ser muy rápido y soportar bastantes cosas sin caerse."

print("Evaluando requerimiento...\n")
resultado = chain.invoke(requerimiento_usuario)
print(resultado)